In [ ]:
# Cell 00: Install
import os, sys, subprocess, random
def pip_install(pkgs):
    if isinstance(pkgs, str): pkgs = [pkgs]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install(["EduData", "pandas", "numpy", "tqdm", "scikit-learn"])
pip_install(["transformers<5.0.0"])


In [ ]:
# Cell 01: Setup
import numpy as np
import pandas as pd
from dataclasses import dataclass
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from EduData.DataSet import get_data
import re
from copy import deepcopy
from torch.utils.data import DataLoader
import torch.nn.functional as F
import subprocess, sys
from collections import defaultdict
import math
import pathlib
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
os.environ["TOKENIZERS_PARALLELISM"] = "false"
SEED = 400101956
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# Cell 02: Download Slepemapy.cz dataset

DATASET = "slepemapy-2015-05-21"
DATA_ROOT = "/kaggle/working/slepemapy" if os.path.exists("/kaggle/working") else "./slepemapy"
os.makedirs(DATA_ROOT, exist_ok=True)

ZIP_URL  = "https://www.fi.muni.cz/adaptivelearning/data/slepemapy/2015-05-21.zip"
ZIP_PATH = os.path.join(DATA_ROOT, "2015-05-21.zip")
EXTRACT_DIR = os.path.join(DATA_ROOT, "2015-05-21")

os.makedirs(EXTRACT_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print("Downloading:", ZIP_URL)
    subprocess.check_call(["bash", "-lc", f"wget -q -O '{ZIP_PATH}' '{ZIP_URL}'"])
else:
    print("Zip already exists:", ZIP_PATH)

need_files = ["answer.csv", "place.csv", "place_type.csv", "README.md"]
missing = [f for f in need_files if not os.path.exists(os.path.join(EXTRACT_DIR, f))]
if missing:
    print("Extracting zip to:", EXTRACT_DIR)
    subprocess.check_call(["bash", "-lc", f"unzip -q -o '{ZIP_PATH}' -d '{EXTRACT_DIR}'"])
else:
    print("Already extracted:", EXTRACT_DIR)

print("Files:", sorted([p.name for p in pathlib.Path(EXTRACT_DIR).glob("*")]))


PLACE_PATH = os.path.join(EXTRACT_DIR, "place.csv")
TYPE_PATH  = os.path.join(EXTRACT_DIR, "place_type.csv")
ANS_PATH   = os.path.join(EXTRACT_DIR, "answer.csv")

place_df = pd.read_csv(PLACE_PATH, sep=";", nrows=5)
ptype_df = pd.read_csv(TYPE_PATH,  sep=";", nrows=10)
ans_df   = pd.read_csv(ANS_PATH,   sep=";", nrows=5)

print("place.csv columns:", list(place_df.columns))
print(place_df.to_string(index=False))

print("\nplace_type.csv columns:", list(ptype_df.columns))
print(ptype_df.to_string(index=False))

print("\nanswer.csv columns:", list(ans_df.columns))
print(ans_df.to_string(index=False))

assert "name" in place_df.columns, "Expected place.csv to contain a 'name' column (place name)."
print("\nPASS: place.csv contains 'name' => we can form question text from place names.")


In [ ]:
# Cell 03: Parse answer.csv 

PLACE_PATH = os.path.join(EXTRACT_DIR, "place.csv")
TYPE_PATH  = os.path.join(EXTRACT_DIR, "place_type.csv")
ANS_PATH   = os.path.join(EXTRACT_DIR, "answer.csv")
place_full = pd.read_csv(PLACE_PATH, sep=";")
ptype_full = pd.read_csv(TYPE_PATH,  sep=";")

place_id_to_name = dict(zip(place_full["id"].astype(int), place_full["name"].astype(str)))
place_id_to_type = dict(zip(place_full["id"].astype(int), place_full["type"].astype(int)))
ptype_id_to_name = dict(zip(ptype_full["id"].astype(int), ptype_full["name"].astype(str)))

def parse_time_to_epoch_seconds(series: pd.Series, fallback_offset_start: int) -> np.ndarray:
    dt = pd.to_datetime(series, errors="coerce")
    good = dt.notna().to_numpy()
    out = np.empty(len(series), dtype=np.float64)
    if good.any():
        ns = dt.to_numpy(dtype="datetime64[ns]").astype("int64")
        out[good] = (ns[good] / 1e9).astype(np.float64)
    bad_idx = np.where(~good)[0]
    if bad_idx.size > 0:
        out[bad_idx] = (fallback_offset_start + bad_idx).astype(np.float64)
    return out

def make_question_text(place_asked: int, qtype: int, place_map):
    asked_name = place_id_to_name.get(int(place_asked), f"place_{int(place_asked)}")
    asked_type_id = place_id_to_type.get(int(place_asked), None)
    asked_type_name = ptype_id_to_name.get(int(asked_type_id), "place") if asked_type_id is not None else "place"

    map_part = ""
    if place_map is not None and int(place_map) >= 0:
        map_name = place_id_to_name.get(int(place_map), f"map_{int(place_map)}")
        map_part = f" (map: {map_name})"

    if int(qtype) == 1:
        return f"Where is {asked_name}?{map_part} Type: locate-on-map."
    else:
        return f"What is the name of the highlighted {asked_type_name}? Answer: {asked_name}.{map_part} Type: name-highlighted-place."

def make_concept_tokens(place_asked: int, qtype: int, place_map):
    toks = set()
    pa = int(place_asked)
    qt = int(qtype)
    toks.add(f"place:{pa}")
    toks.add(f"qtype:{qt}")

    pt = place_id_to_type.get(pa, None)
    if pt is not None:
        toks.add(f"ptype:{int(pt)}")

    if place_map is not None and int(place_map) >= 0:
        toks.add(f"map:{int(place_map)}")
    return toks


CHUNKSIZE = 250_000

usecols = ["user", "place_asked", "place_answered", "type", "inserted", "place_map", "language"]


user_map = {}
q_map = {}                
slep_qtext_by_qid = {}    
q_concepts_tok = {}     

user_arrays, q_arrays, y_arrays, t_arrays = [], [], [], []
offset = 0

reader = pd.read_csv(
    ANS_PATH, sep=";", usecols=usecols, chunksize=CHUNKSIZE, low_memory=True
)

FILTER_LANGUAGE = None  

for chunk in tqdm(reader, desc="Parse Slepemapy answer.csv"):
    chunk = chunk.dropna(subset=["user", "place_asked", "type", "inserted"])

    if FILTER_LANGUAGE is not None and "language" in chunk.columns:
        chunk = chunk[chunk["language"].astype("Int64") == int(FILTER_LANGUAGE)]

    if len(chunk) == 0:
        continue

    u_raw = chunk["user"].astype(str).to_numpy()
    u_codes, u_uniques = pd.factorize(u_raw, sort=False)
    u_l2g = np.empty(len(u_uniques), dtype=np.int32)
    for i, val in enumerate(u_uniques):
        if val in user_map:
            u_l2g[i] = user_map[val]
        else:
            nid = len(user_map)
            user_map[val] = nid
            u_l2g[i] = nid
    u_ids = u_l2g[u_codes]


    pa = chunk["place_asked"].astype("Int64").to_numpy()
    pan = chunk["place_answered"].astype("Int64").to_numpy() 
    correct = (pan == pa) & (~pd.isna(pan))
    yy = correct.astype(np.int8)


    tt = parse_time_to_epoch_seconds(chunk["inserted"], fallback_offset_start=offset)

    qtype = chunk["type"].astype("Int64").to_numpy()
    pmap = chunk["place_map"].to_numpy()  
    pmap_int = np.where(np.isnan(pmap), -1, pmap).astype(np.int64)

    q_ids = np.empty(len(chunk), dtype=np.int32)
    for i in range(len(chunk)):
        key = (int(pa[i]), int(qtype[i]), int(pmap_int[i]))
        qid = q_map.get(key, None)
        if qid is None:
            qid = len(q_map)
            q_map[key] = qid
            slep_qtext_by_qid[qid] = make_question_text(key[0], key[1], key[2])
            q_concepts_tok[qid] = make_concept_tokens(key[0], key[1], key[2])
        q_ids[i] = qid

    offset += len(chunk)

    user_arrays.append(u_ids.astype(np.int32))
    q_arrays.append(q_ids.astype(np.int32))
    y_arrays.append(yy.astype(np.int8))
    t_arrays.append(tt.astype(np.float64))

kt_df = pd.DataFrame({
    "user_id": np.concatenate(user_arrays) if user_arrays else np.empty((0,), dtype=np.int32),
    "question_id": np.concatenate(q_arrays) if q_arrays else np.empty((0,), dtype=np.int32),
    "correct": np.concatenate(y_arrays) if y_arrays else np.empty((0,), dtype=np.int8),
    "time": np.concatenate(t_arrays) if t_arrays else np.empty((0,), dtype=np.float64),
})

kt_df = kt_df.dropna(subset=["user_id", "question_id", "correct", "time"]).copy()
kt_df["correct"] = kt_df["correct"].astype(np.int8).clip(0, 1)

print(kt_df.head())
print("Interactions:", len(kt_df), "Users:", kt_df["user_id"].nunique(), "Questions:", kt_df["question_id"].nunique())

print("\nExample question texts (for LLM semantic encoder):")
for k in sorted(list(slep_qtext_by_qid.keys()))[:5]:
    print(k, "->", slep_qtext_by_qid[k])


In [ ]:
# Cell 04: Build τ (global), nodes, concepts
TAU_MODE = "rank_time"

@dataclass
class KTData:
    src_node_ids: np.ndarray
    dst_node_ids: np.ndarray
    node_interact_times: np.ndarray
    labels: np.ndarray
    edge_ids: np.ndarray

def make_global_tau(df: pd.DataFrame, time_col="time", mode="rank_time") -> pd.DataFrame:
    df = df.reset_index(drop=True).copy()
    if mode == "use_order":
        df["tau"] = np.arange(len(df), dtype=np.float32)
        return df
    df["_row"] = np.arange(len(df), dtype=np.int64)
    order = np.lexsort((df["_row"].values, df[time_col].values))
    tau = np.empty(len(df), dtype=np.int64)
    tau[order] = np.arange(len(df), dtype=np.int64)
    df["tau"] = tau.astype(np.float32)
    return df.drop(columns=["_row"])

kt_df = make_global_tau(kt_df, time_col="time", mode=TAU_MODE)
print("tau(min/max):", float(kt_df["tau"].min()), float(kt_df["tau"].max()))

num_users = int(kt_df["user_id"].max()) + 1 if len(kt_df) else 0
num_questions = int(kt_df["question_id"].max()) + 1 if len(kt_df) else 0

Q_OFFSET = num_users
num_nodes = num_users + num_questions

question_node_ids_all = (Q_OFFSET + np.arange(num_questions, dtype=np.int64)).astype(np.int64)
q2nid = {int(qid): int(Q_OFFSET + qid) for qid in range(num_questions)}
nid2q = {int(Q_OFFSET + qid): int(qid) for qid in range(num_questions)}

print("num_users:", num_users, "num_questions:", num_questions, "num_nodes:", num_nodes)

UNK_CONCEPT = "__unk__"
all_concepts = set()
for qid in range(num_questions):
    toks = q_concepts_tok.get(int(qid), set())
    if not toks:
        continue
    for tok in toks:
        all_concepts.add(str(tok))

concepts = [UNK_CONCEPT] + sorted([c for c in all_concepts if c != UNK_CONCEPT])
c2id = {c: i for i, c in enumerate(concepts)}
unk_id = c2id[UNK_CONCEPT]
num_concepts = len(concepts)

q_concept_sets = {}
for qid in range(num_questions):
    toks = q_concepts_tok.get(int(qid), set())
    ids = []
    seen = set()
    for tok in toks:
        cid = int(c2id.get(str(tok), unk_id))
        if cid not in seen:
            seen.add(cid)
            ids.append(cid)
    q_concept_sets[int(qid)] = sorted(ids) if ids else [unk_id]

q_primary_concept = {q: int(v[0]) for q, v in q_concept_sets.items()}
q_concept_sets_for_cgc = [set(q_concept_sets[int(qid)]) for qid in range(num_questions)]

print("num_concepts:", num_concepts)
print("Example concepts for q0:", [concepts[c] for c in q_concept_sets.get(0, [])[:10]])

src = kt_df["user_id"].to_numpy(dtype=np.int64)
dst = (kt_df["question_id"].to_numpy(dtype=np.int64) + Q_OFFSET)
tau = kt_df["tau"].to_numpy(dtype=np.float32)
y   = kt_df["correct"].to_numpy(dtype=np.int64)
edge_ids = np.arange(len(y), dtype=np.int64)

full_data = KTData(src, dst, tau, y, edge_ids)

TRAIN_FRAC = 0.8
VAL_FRAC   = 0.1
TEST_FRAC  = 0.1

n = len(y)
split = np.zeros(n, dtype=np.int8) 
order = np.lexsort((tau, src))   

i = 0
while i < n:
    u = src[order[i]]
    j = i + 1
    while j < n and src[order[j]] == u:
        j += 1

    m = j - i
    idx_u = order[i:j]

    n_train = int(np.floor(m * TRAIN_FRAC))
    n_val   = int(np.floor(m * VAL_FRAC))

    if m >= 1 and n_train == 0:
        n_train = 1
    if m > 1 and n_train >= m:
        n_train = m - 1

    rem = m - n_train
    max_val = max(0, rem - 1)
    n_val = min(n_val, max_val)

    if n_train < m:
        if n_val > 0:
            split[idx_u[n_train:n_train + n_val]] = 1
        split[idx_u[n_train + n_val:]] = 2

    i = j

train_idx = np.where(split == 0)[0]
val_idx   = np.where(split == 1)[0]
test_idx  = np.where(split == 2)[0]

train_data = KTData(src[train_idx], dst[train_idx], tau[train_idx], y[train_idx], edge_ids[train_idx])
val_data   = KTData(src[val_idx],   dst[val_idx],   tau[val_idx],   y[val_idx],   edge_ids[val_idx])
test_data  = KTData(src[test_idx],  dst[test_idx],  tau[test_idx],  y[test_idx],  edge_ids[test_idx])

print("train/val/test:", len(train_idx), len(val_idx), len(test_idx))
print("ratios:", len(train_idx)/n, len(val_idx)/n, len(test_idx)/n)

question_concept_multi_hot = np.zeros((num_questions, num_concepts), dtype=np.float16)
for qid in range(num_questions):
    for cid in q_concept_sets[int(qid)]:
        question_concept_multi_hot[qid, int(cid)] = 1.0

question_raw_features_base = np.zeros((num_questions, 1), dtype=np.float32)
for qid in range(num_questions):
    question_raw_features_base[qid, 0] = q_primary_concept[int(qid)]

print("question_concept_multi_hot:", question_concept_multi_hot.shape)


In [ ]:
# Cell 05:  DataLoader

BATCH_SIZE = 4096 

train_indices = np.argsort(train_data.node_interact_times)
val_indices   = np.argsort(val_data.node_interact_times)
test_indices  = np.argsort(test_data.node_interact_times)

print("NumPy batching enabled.")
print("BATCH_SIZE:", BATCH_SIZE)
print("train/val/test sizes:", len(train_indices), len(val_indices), len(test_indices))

train_idx_loader = train_indices
val_idx_loader   = val_indices
test_idx_loader  = test_indices


In [ ]:
# Cell 06: Semantic embeddings from Slepemapy question text

TEXT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMB_CACHE = f"./sem_cache_{DATASET}_Q{num_questions}_C{num_concepts}.npy"

@torch.no_grad()
def encode_texts_minilm(texts, batch_size=256, max_len=192):
    tok = AutoTokenizer.from_pretrained(TEXT_MODEL)
    mdl = AutoModel.from_pretrained(TEXT_MODEL).to(device).eval()
    out = []
    for st in tqdm(range(0, len(texts), batch_size), desc="MiniLM encode"):
        batch = texts[st:st+batch_size]
        toks = tok(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
        hs = mdl(**toks).last_hidden_state
        mask = toks["attention_mask"].unsqueeze(-1).float()
        pooled = (hs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        pooled = F.normalize(pooled, dim=1)
        out.append(pooled.detach().cpu().numpy())
    return np.vstack(out).astype(np.float32)

texts = []
for qid in range(num_questions):
    q_text = str(slep_qtext_by_qid.get(int(qid), f"Q{qid}"))

    cids = q_concept_sets.get(int(qid), [])
    cnames = [concepts[int(c)] for c in cids[:30]]
    texts.append(f"Question: {q_text} Skills: {' '.join(cnames)}")

if os.path.exists(EMB_CACHE):
    q_emb = np.load(EMB_CACHE).astype(np.float32)
    if q_emb.shape[0] != num_questions:
        raise ValueError(f"Cached embeddings mismatch: {q_emb.shape} vs {num_questions}. Delete {EMB_CACHE}.")
    print("Loaded semantic embeddings:", EMB_CACHE, q_emb.shape)
else:
    q_emb = encode_texts_minilm(texts)
    np.save(EMB_CACHE, q_emb)
    print("Saved semantic embeddings:", EMB_CACHE, q_emb.shape)

question_sem_features = q_emb.astype(np.float32)
Dsem = question_sem_features.shape[1]
print("question_sem_features:", question_sem_features.shape)

node_sem_features = np.zeros((num_nodes, Dsem), dtype=np.float32)
node_concept_multi_hot = np.zeros((num_nodes, num_concepts), dtype=np.float16)

node_sem_features[Q_OFFSET:Q_OFFSET + num_questions] = question_sem_features
node_concept_multi_hot[Q_OFFSET:Q_OFFSET + num_questions] = question_concept_multi_hot.astype(np.float16)

print("node_sem_features:", node_sem_features.shape)
print("node_concept_multi_hot:", node_concept_multi_hot.shape)


In [ ]:
# Cell 06B: Streaming question metadata store

class QuestionMetaStore:
    """
    Provides sem_cb(q_global_id) and concept_cb(q_global_id) for CGC.
    - Known questions: served from base arrays (q_emb, q_concept_sets).
    - Unseen questions: YOU must register once via register_unseen(...).
    """
    def __init__(self, q_offset, base_sem, base_q_concept_sets, num_concepts, unk_id=0):
        self.q_offset = int(q_offset)
        self.base_sem = np.asarray(base_sem, dtype=np.float32)  
        self.base_q_concept_sets = base_q_concept_sets         
        self.num_concepts = int(num_concepts)
        self.unk_id = int(unk_id)

        self.extra_sem = {}      
        self.extra_concepts = {} 

    def _q_local(self, q_global):
        return int(q_global) - self.q_offset

    def register_unseen(self, q_global_id, sem_vec, concept_ids):
        sem_vec = np.asarray(sem_vec, dtype=np.float32).reshape(-1)
        sem_vec = sem_vec / (np.linalg.norm(sem_vec) + 1e-12)

        cset = set()
        for c in concept_ids:
            ci = int(c)
            if 0 <= ci < self.num_concepts:
                cset.add(ci)
            else:
                cset.add(self.unk_id)

        self.extra_sem[int(q_global_id)] = sem_vec
        self.extra_concepts[int(q_global_id)] = cset

    def get_sem(self, q_global_id):
        qg = int(q_global_id)
        ql = self._q_local(qg)
        if 0 <= ql < self.base_sem.shape[0]:
            return self.base_sem[ql]
        if qg in self.extra_sem:
            return self.extra_sem[qg]
        raise KeyError(
            f"Unseen question {qg} has no registered semantic embedding. "
            f"Call meta_store.register_unseen(q_global_id, sem_vec, concept_ids) first."
        )

    def get_concepts(self, q_global_id):
        qg = int(q_global_id)
        ql = self._q_local(qg)
        if 0 <= ql < self.base_sem.shape[0]:
            return set(int(x) for x in self.base_q_concept_sets.get(ql, [self.unk_id]))
        if qg in self.extra_concepts:
            return set(self.extra_concepts[qg])
        raise KeyError(
            f"Unseen question {qg} has no registered concepts. "
            f"Call meta_store.register_unseen(q_global_id, sem_vec, concept_ids) first."
        )


unk_id = c2id["__unk__"]
meta_store = QuestionMetaStore(
    q_offset=Q_OFFSET,
    base_sem=q_emb,
    base_q_concept_sets=q_concept_sets,
    num_concepts=num_concepts,
    unk_id=unk_id
)

SEM_CB = lambda q_global: meta_store.get_sem(q_global)
CONCEPT_CB = lambda q_global: meta_store.get_concepts(q_global)


In [ ]:
# Cell 07: Hyperparams 
# # temporal decay + pruning
# W_WINDOW       = 100
# HALF_LIFE      = 80
# TIME_DECAY     = float(np.log(2.0) / HALF_LIFE)
# PRUNE_INTERVAL = 50
# EPS_RATIO   = 0.75
# # sparse CGC graph params
# EDGE_MIN_W  = 0.005   
# MAX_DEGREE  = 64        # richer question neighborhood
# MAX_ANCHORS = 48    
# # contrastive + encoder
# LAMBDA_SCL  = 0.4   
# LAMBDA_SELF = 0.3
# TAU_SCL     = 0.07     
# BETA_STAGE  = 0.02      # stronger stage penalty when W_WINDOW is larger
# MAX_SEQ_LEN = 300
# CGC_BUILD_BLOCK = 256


######################################################################

# ---- SET 1: Balanced baseline (good first run) ----

W_WINDOW       = 50
HALF_LIFE      = 80
TIME_DECAY     = float(np.log(2.0) / HALF_LIFE)
PRUNE_INTERVAL = 32
EPS_RATIO      = 0.75   

EDGE_MIN_W     = 0.005
MAX_DEGREE     = 40       
MAX_ANCHORS    = 30 

LAMBDA_SCL  = 1
TAU_SCL     = 0.1
BETA_STAGE  = 0.03

LAMBDA_SELF = 1
MAX_SEQ_LEN = 32 
CGC_BUILD_BLOCK = 256


# train_CaRe-KT: lr=1e-3, dim=128

#################################################################################

# # ---- SET 2: Larger context / stronger graph (often best if GPU allows) ----
# W_WINDOW       = 200
# HALF_LIFE      = 120
# TIME_DECAY     = float(np.log(2.0) / HALF_LIFE)
# PRUNE_INTERVAL = 100
# EPS_RATIO      = 0.70

# EDGE_MIN_W  = 0.003
# MAX_DEGREE  = 96
# MAX_ANCHORS = 64

# LAMBDA_SCL  = 0.50
# LAMBDA_SELF = 0.25
# TAU_SCL     = 0.05
# BETA_STAGE  = 0.015

# MAX_SEQ_LEN = 400
# CGC_BUILD_BLOCK = 384

# # train_CaRe-KT: lr=8e-4, dim=160

#################################################################################

# # ---- SET 3: More regularized / lighter & faster (use if overfitting or slow) ----
# W_WINDOW       = 60
# HALF_LIFE      = 50
# TIME_DECAY     = float(np.log(2.0) / HALF_LIFE)
# PRUNE_INTERVAL = 30
# EPS_RATIO      = 0.80

# EDGE_MIN_W  = 0.008
# MAX_DEGREE  = 48
# MAX_ANCHORS = 32

# LAMBDA_SCL  = 0.30
# LAMBDA_SELF = 0.35
# TAU_SCL     = 0.10
# BETA_STAGE  = 0.03

# MAX_SEQ_LEN = 200
# CGC_BUILD_BLOCK = 256

# # train_reakt: lr=1.5e-3, dim=96
#################################################################################


In [ ]:
# Cell 08: ContextGraphCache
# (1) CGC graph at time t contains ONLY "active" (observed within window) questions.
#     Nodes/edges are activated on first observation (query-time) and can be deactivated by pruning.
# (2) When a node’s buffer becomes empty after pruning, we remove it from the active graph:
#     deactivate -> clear edges to/from it (so Stage-1 doesn't scan it).
# - We keep base question embeddings/concepts loaded, but they are INACTIVE until observed.
# - We precompute candidate neighbor lists once (top Kcand) to avoid full O(Q^2) scans per activation.

class ContextGraphCache(nn.Module):
    def __init__(
        self,
        question_node_ids,         
        question_embeddings,      
        question_concept_sets,    
        device,
        time_decay=TIME_DECAY,
        window_size=W_WINDOW,
        prune_interval=PRUNE_INTERVAL,
        epsilon_ratio=EPS_RATIO,
        edge_min_w=EDGE_MIN_W,
        max_degree=MAX_DEGREE,
        max_anchors=MAX_ANCHORS,
        build_block=CGC_BUILD_BLOCK,
        concept_dim=None,          
        sem_cb=None,               
        concept_cb=None,           
        candidate_factor=6,        
        activate_at_init=False,   
    ):
        super().__init__()
        self.device = device

        self.sem_cb = sem_cb
        self.concept_cb = concept_cb

        self.beta0 = nn.Parameter(torch.tensor(0.5, device=device))
        self.beta1 = nn.Parameter(torch.tensor(0.2, device=device))

        self.time_decay = float(time_decay)
        self.window_size = float(window_size)
        self.prune_interval = int(prune_interval)
        self.epsilon_ratio = float(epsilon_ratio)
        self.edge_min_w = float(edge_min_w)
        self.max_degree = int(max_degree)
        self.max_anchors = int(max_anchors)
        self.build_block = int(build_block)

        qids = list(map(int, list(question_node_ids)))
        self.question_ids = qids[:]                    
        self.qid2local = {q: i for i, q in enumerate(qids)}
        self.Q_base = len(self.question_ids)

        E = np.asarray(question_embeddings, dtype=np.float32)
        E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
        self.E = E                                    

        assert len(question_concept_sets) == self.Q_base
        self.Csets = [set(map(int, s)) for s in question_concept_sets]  

        if concept_dim is not None:
            self.C = int(concept_dim)
        else:
            cmax = -1
            for s in self.Csets:
                if s:
                    cmax = max(cmax, max(s))
            self.C = max(cmax + 1, 1)

        self.active = np.zeros((self.Q_base,), dtype=np.bool_)      
        self.last_time = np.full((self.Q_base,), -1.0, dtype=np.float32)

        self.buffers = defaultdict(list)
        self.latest_time = None
        self.update_counter = 0

        self.adj = [dict() for _ in range(self.Q_base)]


        self.Kcand = int(max(0, self.max_degree) * int(candidate_factor)) if self.max_degree > 0 else 0
        self.Kcand = int(min(self.Kcand, 256)) 

        self.cand_len = np.zeros((self.Q_base,), dtype=np.int16)
        if self.Kcand > 0:
            self.cand_j   = np.full((self.Q_base, self.Kcand), -1, dtype=np.int32)
            self.cand_cos = np.zeros((self.Q_base, self.Kcand), dtype=np.float16)
            self.cand_jac = np.zeros((self.Q_base, self.Kcand), dtype=np.float16)
            self.cand_ov  = np.zeros((self.Q_base, self.Kcand), dtype=np.float16)
            self._build_candidates_once()
        else:
            self.cand_j = None
            self.cand_cos = None
            self.cand_jac = None
            self.cand_ov = None

        if activate_at_init:
            for i in range(self.Q_base):
                self._activate_local(i)


    def _build_candidates_once(self):
        """
        Build top-Kcand candidate lists for base questions.
        This DOES NOT build the active graph; it only caches potential neighbors.
        """
        Q = self.Q_base
        C = self.C

        M = np.zeros((Q, C), dtype=np.uint8)
        for i, s in enumerate(self.Csets):
            for c in s:
                if 0 <= c < C:
                    M[i, c] = 1
        Mall = M.astype(np.float32)

        with torch.no_grad():
            beta0 = float(self.beta0.detach().cpu().item())
            beta1 = float(self.beta1.detach().cpu().item())

        def sigmoid(x):
            return 1.0 / (1.0 + np.exp(-x))

        for st in tqdm(range(0, Q, self.build_block), desc="CGC build candidates"):
            ed = min(Q, st + self.build_block)
            Eb = self.E[st:ed]
            Mb = M[st:ed].astype(np.float32)

            cos = Eb @ self.E.T
            inter = Mb @ Mall.T
            card_b = Mb.sum(axis=1, keepdims=True)
            card_all = Mall.sum(axis=1, keepdims=True).T
            union = card_b + card_all - inter
            jacc = inter / (union + 1e-8)
            overlap = (2.0 * inter) / (card_b + card_all + 1e-8)

            alpha = sigmoid(beta0 - beta1 * overlap)
            w = alpha * cos + (1.0 - alpha) * jacc

            for bi in range(ed - st):
                i = st + bi
                w[bi, i] = -1e9 

                if self.Kcand <= 0:
                    continue
                kk = min(self.Kcand, Q - 1)
                idx = np.argpartition(w[bi], -kk)[-kk:]
                idx = idx[np.argsort(w[bi, idx])[::-1]] 

                self.cand_len[i] = np.int16(len(idx))
                self.cand_j[i, :len(idx)] = idx.astype(np.int32)
                self.cand_cos[i, :len(idx)] = cos[bi, idx].astype(np.float16)
                self.cand_jac[i, :len(idx)] = jacc[bi, idx].astype(np.float16)
                self.cand_ov[i, :len(idx)]  = overlap[bi, idx].astype(np.float16)


    @staticmethod
    def _w_static_detached(cos_ij: float, jacc_ij: float, ov_ij: float, beta0: float, beta1: float) -> float:
        a = 1.0 / (1.0 + math.exp(-(beta0 - beta1 * ov_ij)))
        return a * cos_ij + (1.0 - a) * jacc_ij

    def _cap_degree_single(self, i: int):
        if self.max_degree <= 0:
            return
        nbrs = list(self.adj[i].items())
        if len(nbrs) <= self.max_degree:
            return

        with torch.no_grad():
            beta0 = float(self.beta0.detach().cpu().item())
            beta1 = float(self.beta1.detach().cpu().item())

        scores = np.asarray(
            [self._w_static_detached(vals[0], vals[1], vals[2], beta0, beta1) for (_, vals) in nbrs],
            dtype=np.float32
        )
        keep_idx = np.argpartition(scores, -self.max_degree)[-self.max_degree:]
        keep_set = set(int(nbrs[k][0]) for k in keep_idx.tolist())

        for j, _ in nbrs:
            if int(j) not in keep_set:
                self.adj[i].pop(int(j), None)
                self.adj[int(j)].pop(i, None)

    def _deactivate_local(self, ql: int):
        """Remove node from active CGC graph: clear its edges and mark inactive."""
        if ql < 0 or ql >= len(self.active):
            return
        if not bool(self.active[ql]):
            return

        for j in list(self.adj[ql].keys()):
            self.adj[int(j)].pop(int(ql), None)
        self.adj[ql].clear()

        self.active[ql] = False
        self.last_time[ql] = np.float32(-1.0)
        self.buffers.pop(int(ql), None)

    def _activate_local(self, ql: int):
        """Add node to active CGC graph and connect it to already-active nodes via cached candidates."""
        if ql < 0 or ql >= self.Q_base:
            return
        if bool(self.active[ql]):
            return

        self.active[ql] = True

        if self.Kcand <= 0:
            return

        L = int(self.cand_len[ql])
        if L <= 0:
            return

        js = self.cand_j[ql, :L].astype(np.int64)
        mask = self.active[js]
        js = js[mask]
        if js.size == 0:
            return

        cos = self.cand_cos[ql, :L].astype(np.float32)[mask]
        jac = self.cand_jac[ql, :L].astype(np.float32)[mask]
        ov  = self.cand_ov[ql, :L].astype(np.float32)[mask]

        with torch.no_grad():
            beta0 = float(self.beta0.detach().cpu().item())
            beta1 = float(self.beta1.detach().cpu().item())

        w = np.asarray(
            [self._w_static_detached(float(c), float(j), float(o), beta0, beta1) for c, j, o in zip(cos, jac, ov)],
            dtype=np.float32
        )

        keep = np.where(w >= self.edge_min_w)[0] if self.edge_min_w > 0 else np.arange(len(w))
        if keep.size == 0:
            return

        if self.max_degree > 0 and keep.size > self.max_degree:
            top = np.argpartition(w[keep], -self.max_degree)[-self.max_degree:]
            keep = keep[top]

        for k in keep.tolist():
            j = int(js[k])
            self.adj[ql][j] = (float(cos[k]), float(jac[k]), float(ov[k]))
            self.adj[j][ql] = (float(cos[k]), float(jac[k]), float(ov[k]))

        self._cap_degree_single(ql)
        for j in list(self.adj[ql].keys()):
            self._cap_degree_single(int(j))

    def reset_state(self):
        """
        Full reset to time 0:
        - clear buffers
        - deactivate ALL nodes (so CGC starts empty -> observed-up-to-t)
        """
        self.buffers.clear()
        self.latest_time = None
        self.update_counter = 0

        for i in range(self.Q_base):
            if self.adj[i]:
                for j in list(self.adj[i].keys()):
                    self.adj[int(j)].pop(i, None)
                self.adj[i].clear()

        self.active[:] = False
        self.last_time[:] = np.float32(-1.0)

    def has_question(self, q_global_id: int) -> bool:
        return int(q_global_id) in self.qid2local

    def get_sem_and_concepts(self, q_global_id: int):
        qg = int(q_global_id)
        ql = self.qid2local.get(qg, None)
        if ql is None:
            return None, None
        return self.E[int(ql)], self.Csets[int(ql)]

    def _auto_register_if_needed(self, q_global_id: int):
        """
        For truly unseen questions (not in base list), register dynamically.
        """
        qg = int(q_global_id)
        if qg in self.qid2local:
            return

        if self.sem_cb is None or self.concept_cb is None:
            return

        sem = np.asarray(self.sem_cb(qg), dtype=np.float32).reshape(-1)
        sem = sem / (np.linalg.norm(sem) + 1e-12)
        cset = set(int(x) for x in self.concept_cb(qg))
        if self.C is not None and self.C > 0:
            cset = set(x for x in cset if 0 <= x < self.C) or set()

        ql_new = self.Q_base
        self.question_ids.append(qg)
        self.qid2local[qg] = ql_new
        self.E = np.vstack([self.E, sem[None, :]])
        self.Csets.append(cset)

        self.active = np.append(self.active, np.bool_(False))
        self.last_time = np.append(self.last_time, np.float32(-1.0))
        self.adj.append(dict())

        self.Q_base += 1

    def _activate_dynamic(self, ql: int):
        """
        Activation path for dynamic nodes: exact scan over active nodes.
        """
        if bool(self.active[ql]):
            return
        self.active[ql] = True

        act = np.where(self.active)[0]
        act = act[act != ql]
        if act.size == 0:
            return

        with torch.no_grad():
            beta0 = float(self.beta0.detach().cpu().item())
            beta1 = float(self.beta1.detach().cpu().item())

        sem = self.E[ql]
        cos_all = (self.E[act] @ sem).astype(np.float32)

        a_set = self.Csets[ql]
        jac = np.zeros(act.size, dtype=np.float32)
        ov  = np.zeros(act.size, dtype=np.float32)
        for ii, j in enumerate(act.tolist()):
            b_set = self.Csets[int(j)]
            if not a_set and not b_set:
                jac[ii] = 0.0; ov[ii] = 0.0
            else:
                inter = len(a_set & b_set)
                union = len(a_set | b_set)
                jac[ii] = float(inter) / float(union + 1e-12)
                ov[ii]  = float(2.0 * inter) / float((len(a_set) + len(b_set)) + 1e-12)

        w = np.asarray(
            [self._w_static_detached(float(c), float(j), float(o), beta0, beta1) for c, j, o in zip(cos_all, jac, ov)],
            dtype=np.float32
        )
        keep = np.where(w >= self.edge_min_w)[0] if self.edge_min_w > 0 else np.arange(len(w))
        if keep.size == 0:
            return

        if self.max_degree > 0 and keep.size > self.max_degree:
            top = np.argpartition(w[keep], -self.max_degree)[-self.max_degree:]
            keep = keep[top]

        for k in keep.tolist():
            j = int(act[k])
            self.adj[ql][j] = (float(cos_all[k]), float(jac[k]), float(ov[k]))
            self.adj[j][ql] = (float(cos_all[k]), float(jac[k]), float(ov[k]))

        self._cap_degree_single(ql)
        for j in list(self.adj[ql].keys()):
            self._cap_degree_single(int(j))

    def _ensure_active(self, q_global_id: int):
        qg = int(q_global_id)
        self._auto_register_if_needed(qg)
        ql = self.qid2local.get(qg, None)
        if ql is None:
            return None
        ql = int(ql)
        if ql < self.Q_base:
            if ql < len(self.cand_len):
                self._activate_local(ql)
            else:
                self._activate_dynamic(ql)
        else:
            self._activate_dynamic(ql)
        return ql

    def _prune_buffers(self):
        if self.latest_time is None:
            return
        cutoff = self.latest_time - self.window_size

        for ql, buf in list(self.buffers.items()):
            nb = [x for x in buf if x[1] >= cutoff]
            if nb:
                self.buffers[ql] = nb
                self.last_time[ql] = np.float32(max(t for (_, t, _) in nb))
            else:
                self._deactivate_local(int(ql))

    def _prune_edges_by_threshold_and_cap_degree(self):
        if self.edge_min_w <= 0.0 and self.max_degree <= 0:
            return

        with torch.no_grad():
            beta0 = float(self.beta0.detach().cpu().item())
            beta1 = float(self.beta1.detach().cpu().item())

        if self.edge_min_w > 0.0:
            for i in range(self.Q_base):
                if not bool(self.active[i]):
                    continue
                for j in list(self.adj[i].keys()):
                    if int(j) < i:
                        continue
                    cos_ij, jac_ij, ov_ij = self.adj[i][int(j)]
                    w = self._w_static_detached(cos_ij, jac_ij, ov_ij, beta0, beta1)
                    if w < self.edge_min_w:
                        self.adj[i].pop(int(j), None)
                        self.adj[int(j)].pop(i, None)

        if self.max_degree > 0:
            for i in range(self.Q_base):
                if bool(self.active[i]):
                    self._cap_degree_single(i)

    def update(self, src_ids, dst_ids, times, labels):
        srcs = np.asarray(src_ids).astype(np.int64)
        dsts = np.asarray(dst_ids).astype(np.int64)
        ts   = np.asarray(times).astype(np.float32)
        ys   = np.asarray(labels).astype(np.int64)

        for s, q, t, y in zip(srcs, dsts, ts, ys):
            q = int(q)
            ql = self._ensure_active(q)
            if ql is None:
                continue

            t = float(t)
            self.latest_time = t if self.latest_time is None else max(self.latest_time, t)

            self.buffers[int(ql)].append((int(s), t, int(y)))
            self.last_time[int(ql)] = np.float32(t)

        self.update_counter += len(srcs)
        while self.update_counter >= self.prune_interval:
            self._prune_buffers()
            self._prune_edges_by_threshold_and_cap_degree()
            self.update_counter -= self.prune_interval

    def get_batch_anchors(self, target_q_ids, target_times, return_debug=False):
        tq = np.asarray(target_q_ids).astype(np.int64)
        tt = np.asarray(target_times).astype(np.float32)

        batch_anchors, batch_debug = [], []

        beta0_val = float(self.beta0.detach().cpu().item())
        beta1_val = float(self.beta1.detach().cpu().item())

        def sigmoid(x):
            return 1.0 / (1.0 + np.exp(-x))

        for q_id, tau_t in zip(tq, tt):
            q_id = int(q_id)
            tau_t = float(tau_t)

            ql_t = self._ensure_active(q_id)
            if ql_t is None:
                batch_anchors.append([])
                if return_debug:
                    batch_debug.append({"stage1_examined": 0})
                continue

            nbrs = list(self.adj[int(ql_t)].items())
            dbg = {"stage1_examined": int(len(nbrs))} if return_debug else None

            if not nbrs:
                batch_anchors.append([])
                if return_debug:
                    batch_debug.append(dbg)
                continue

            dyn_scores = []
            cand = [] 

            for ql_j, (cos_ij, jac_ij, ov_ij) in nbrs:
                tau_max = float(self.last_time[int(ql_j)])
                if tau_max < 0.0:
                    continue
                if tau_max >= tau_t:
                    continue

                a = sigmoid(beta0_val - beta1_val * float(ov_ij))
                w_static = a * float(cos_ij) + (1.0 - a) * float(jac_ij)

                decay = float(np.exp(np.float32(-self.time_decay * abs(tau_t - tau_max))))
                w_dyn = w_static * decay

                dyn_scores.append(w_dyn)
                cand.append((int(ql_j), float(w_static), float(cos_ij), float(jac_ij), float(ov_ij)))

            if not dyn_scores:
                batch_anchors.append([])
                if return_debug:
                    batch_debug.append(dbg)
                continue

            max_w = max(dyn_scores)
            if max_w <= 0.0:
                batch_anchors.append([])
                if return_debug:
                    batch_debug.append(dbg)
                continue

            eps_t = self.epsilon_ratio * max_w
            kept = [c for c, w_dyn in zip(cand, dyn_scores) if w_dyn >= eps_t]
            if not kept:
                batch_anchors.append([])
                if return_debug:
                    batch_debug.append(dbg)
                continue

            anchors = []
            for ql_j, w_static_f, cos_ij, jac_ij, ov_ij in kept:
                buf = self.buffers.get(int(ql_j), [])
                if not buf:
                    continue

                cos_t = torch.tensor(cos_ij, device=self.device)
                jac_t = torch.tensor(jac_ij, device=self.device)
                ov_t  = torch.tensor(ov_ij,  device=self.device)
                a_t = torch.sigmoid(self.beta0 - self.beta1 * ov_t)
                w_static_t = a_t * cos_t + (1.0 - a_t) * jac_t

                for (s_hist, tau_k, y_hist) in reversed(buf):
                    tau_k = float(tau_k)
                    if tau_k >= tau_t:
                        continue

                    decay = float(np.exp(np.float32(-self.time_decay * (tau_t - tau_k))))
                    w_val = w_static_f * decay
                    if w_val < eps_t:
                        break

                    w_t = w_static_t * decay
                    anchors.append({
                        "student_id": int(s_hist),
                        "question_id": int(self.question_ids[int(ql_j)]),
                        "timestamp": float(tau_k),
                        "label": int(y_hist),
                        "weight": w_t,
                        "weight_val": float(w_val),
                    })

            anchors.sort(key=lambda x: x["weight_val"], reverse=True)
            if self.max_anchors > 0 and len(anchors) > self.max_anchors:
                anchors = anchors[:self.max_anchors]
            for a in anchors:
                a.pop("weight_val", None)

            batch_anchors.append(anchors)
            if return_debug:
                batch_debug.append(dbg)

        return (batch_anchors, batch_debug) if return_debug else batch_anchors


In [ ]:
# Cell 09: CaRe-KT 

class StudentMemoryIndex:
    """
    Stores interactions grouped by student in contiguous slices.
    Internally sorts by (student_id, tau) to guarantee contiguity and monotonicity.
    """
    def __init__(self, data: KTData, num_users: int):
        self.num_users = int(num_users)

        src = np.asarray(data.src_node_ids, dtype=np.int64)
        dst = np.asarray(data.dst_node_ids, dtype=np.int64)
        tau = np.asarray(data.node_interact_times, dtype=np.float32)
        y   = np.asarray(data.labels, dtype=np.int8)

        order = np.lexsort((tau, src))
        self.src = src[order]
        self.dst = dst[order]
        self.tau = tau[order]
        self.y   = y[order]

        counts = np.bincount(self.src, minlength=self.num_users).astype(np.int64)
        self.offsets = np.zeros(self.num_users + 1, dtype=np.int64)
        self.offsets[1:] = np.cumsum(counts)

    def prefix(self, student_id: int, cutoff_tau: float, max_len=None):
        s = int(student_id)
        if s < 0 or s >= self.num_users:
            return (np.empty((0,), dtype=np.int64), np.empty((0,), dtype=np.int8))

        st = int(self.offsets[s])
        ed = int(self.offsets[s + 1])
        if ed <= st:
            return (np.empty((0,), dtype=np.int64), np.empty((0,), dtype=np.int8))

        times = self.tau[st:ed]
        k = int(np.searchsorted(times, float(cutoff_tau), side="left"))
        if k <= 0:
            return (np.empty((0,), dtype=np.int64), np.empty((0,), dtype=np.int8))

        if max_len is not None and k > int(max_len):
            k0 = k - int(max_len)
        else:
            k0 = 0

        qs = self.dst[st + k0: st + k]
        rs = self.y[st + k0: st + k]
        return (qs, rs)


class ReAKT(nn.Module):
    def __init__(
        self,
        question_sem_features,
        question_concept_multi_hot,
        q_offset,
        cgc: ContextGraphCache,
        mem_train: StudentMemoryIndex,
        mem_val: StudentMemoryIndex,
        mem_test: StudentMemoryIndex,
        device,
        dim=128,
        lambda_scl=LAMBDA_SCL,
        lambda_self=LAMBDA_SELF,
        tau_scl=TAU_SCL,
        beta_stage=BETA_STAGE,
        max_seq_len=MAX_SEQ_LEN,
        train_max_anchors=8,   
    ):
        super().__init__()
        self.device = device
        self.dim = int(dim)
        self.lambda_scl = float(lambda_scl)
        self.lambda_self = float(lambda_self)
        self.tau_scl = float(tau_scl)
        self.beta_stage = float(beta_stage)
        self.max_seq_len = int(max_seq_len)

        self.train_max_anchors = int(train_max_anchors)

        self.q_offset = int(q_offset)
        self.num_questions = int(question_sem_features.shape[0])

        self.q_sem = torch.from_numpy(np.asarray(question_sem_features)).float().to(device)
        self.q_con = torch.from_numpy(np.asarray(question_concept_multi_hot)).float().to(device)

        self.sem_proj = nn.Linear(self.q_sem.shape[1], dim, bias=False)
        self.con_proj = nn.Linear(self.q_con.shape[1], dim, bias=False)
        self.W_s = nn.Linear(dim, dim, bias=False)
        self.W_c = nn.Linear(dim, dim, bias=False)

        self.resp_embed = nn.Embedding(2, dim)
        self.gru = nn.GRU(input_size=dim * 2, hidden_size=dim, batch_first=True)

        self.W_query_proj = nn.Linear(dim * 2, dim)
        self.query_norm = nn.LayerNorm(dim)

        self.W_gat = nn.Linear(dim, dim, bias=False)
        self.a_gat = nn.Linear(dim * 2, 1, bias=False)
        self.W_prime = nn.Linear(dim, dim)

        self.self_proj = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))
        self.self_pred = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))

        self.predictor = nn.Sequential(
            nn.Linear(dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

        self.aux_loss = torch.tensor(0.0, device=device)

        self.cgc = cgc
        self.mem_train = mem_train
        self.mem_val = mem_val
        self.mem_test = mem_test
        self.memory_mode = "train"

    def set_memory_mode(self, mode: str):
        assert mode in {"train", "val", "test"}
        self.memory_mode = mode

    def reset_state(self):
        self.cgc.reset_state()

    def question_repr(self, q_ids_global):
        qg = torch.as_tensor(q_ids_global, dtype=torch.long, device=self.device).view(-1)
        ql = qg - self.q_offset
        out = torch.zeros(qg.size(0), self.dim, device=self.device)

        mask_known = (ql >= 0) & (ql < self.num_questions)
        if mask_known.any():
            ql_valid = ql[mask_known]
            e_sem = self.sem_proj(self.q_sem[ql_valid])
            e_con = self.con_proj(self.q_con[ql_valid])
            out[mask_known] = self.W_s(e_sem) + self.W_c(e_con)

        idx_unknown = torch.where(~mask_known)[0]
        if idx_unknown.numel() > 0:
            C = int(self.q_con.shape[1])
            for idx in idx_unknown.tolist():
                q_id = int(qg[idx].item())
                sem_np, cset = self.cgc.get_sem_and_concepts(q_id)
                if sem_np is None:
                    continue

                sem_t = torch.from_numpy(np.asarray(sem_np, dtype=np.float32)).to(self.device).view(1, -1)
                con_t = torch.zeros((1, C), device=self.device, dtype=torch.float32)
                for cid in (cset or []):
                    ci = int(cid)
                    if 0 <= ci < C:
                        con_t[0, ci] = 1.0

                e_sem = self.sem_proj(sem_t)
                e_con = self.con_proj(con_t)
                out[idx] = (self.W_s(e_sem) + self.W_c(e_con)).view(-1)

        return out

    def get_subsequence(self, student_id, cutoff_tau):
        s = int(student_id)
        ct = float(cutoff_tau)
        L = self.max_seq_len

        qs_t, rs_t = self.mem_train.prefix(s, ct, max_len=L)
        if self.memory_mode == "train":
            return (qs_t, rs_t)

        qs_v, rs_v = self.mem_val.prefix(s, ct, max_len=None)
        if self.memory_mode == "val":
            need = int(L) if L is not None else None
            if need is None:
                return (np.concatenate([qs_t, qs_v]), np.concatenate([rs_t, rs_v]))
            if len(qs_v) >= need:
                return (qs_v[-need:], rs_v[-need:])
            take_train = need - len(qs_v)
            qs_t2, rs_t2 = self.mem_train.prefix(s, ct, max_len=take_train)
            return (np.concatenate([qs_t2, qs_v]), np.concatenate([rs_t2, rs_v]))

        qs_te, rs_te = self.mem_test.prefix(s, ct, max_len=None)
        need = int(L) if L is not None else None
        if need is None:
            return (np.concatenate([qs_t, qs_v, qs_te]), np.concatenate([rs_t, rs_v, rs_te]))

        if len(qs_te) >= need:
            return (qs_te[-need:], rs_te[-need:])

        rem = need - len(qs_te)
        if len(qs_v) >= rem:
            return (np.concatenate([qs_v[-rem:], qs_te]), np.concatenate([rs_v[-rem:], rs_te]))

        rem2 = rem - len(qs_v)
        qs_t2, rs_t2 = self.mem_train.prefix(s, ct, max_len=rem2)
        return (np.concatenate([qs_t2, qs_v, qs_te]), np.concatenate([rs_t2, rs_v, rs_te]))

    def batch_encode(self, list_of_seqs):
        lengths = [len(s[0]) for s in list_of_seqs]
        max_len = max(lengths) if lengths else 0
        if max_len == 0:
            return torch.zeros(len(list_of_seqs), self.dim, device=self.device)

        B = len(list_of_seqs)
        x = torch.zeros(B, max_len, self.dim * 2, device=self.device)

        for i, (qs, rs) in enumerate(list_of_seqs):
            if len(qs) == 0:
                continue
            vq = self.question_repr(qs)
            eres = self.resp_embed(torch.tensor(rs, dtype=torch.long, device=self.device))
            x[i, :len(qs), :] = torch.cat([vq, eres], dim=1)

        lengths_cpu = torch.tensor([max(1, l) for l in lengths], dtype=torch.long).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)
        return h[-1]

    def augment_sequence(self, seq, drop_prob=0.1):
        qs, rs = seq
        if len(qs) < 2:
            return seq
        keep = (np.random.rand(len(qs)) > drop_prob)
        if keep.sum() == 0:
            return seq
        return (qs[keep], rs[keep])

    def forward(self, src_ids, dst_ids, times, labels=None):
        src = np.asarray(src_ids).astype(np.int64)
        dst = np.asarray(dst_ids).astype(np.int64)
        t   = np.asarray(times).astype(np.float32)

        if self.training:
            y = np.asarray(labels.detach().cpu().numpy() if torch.is_tensor(labels) else labels).astype(np.int64)
            order = np.argsort(t)
            src, dst, t, y = src[order], dst[order], t[order], y[order]
        else:
            y = None

        B = len(src)
        self.aux_loss = torch.tensor(0.0, device=self.device)

        vq_dst = self.question_repr(dst)

        target_hist = []
        aug1, aug2 = [], []
        peer_seqs, peer_w, peer_y, peer_t = [], [], [], []
        peer_counts = []

        for i in range(B):
            s_i = int(src[i]); q_i = int(dst[i]); t_i = float(t[i])

            anchors = self.cgc.get_batch_anchors([q_i], [t_i])[0]

            if self.training and self.train_max_anchors > 0 and len(anchors) > self.train_max_anchors:
                anchors = anchors[:self.train_max_anchors]

            hist = self.get_subsequence(s_i, t_i)
            target_hist.append(hist)

            if self.training:
                aug1.append(self.augment_sequence(hist))
                aug2.append(self.augment_sequence(hist))

            cnt = 0
            for anc in anchors:
                ph = self.get_subsequence(anc["student_id"], anc["timestamp"])
                if len(ph[0]) > 0:
                    peer_seqs.append(ph)
                    peer_w.append(anc["weight"])
                    peer_y.append(anc["label"])
                    peer_t.append(anc["timestamp"])
                    cnt += 1
            peer_counts.append(cnt)

            if self.training:
                self.cgc.update([s_i], [q_i], [t_i], [int(y[i])])

        h_q = self.batch_encode(target_hist)
        z_q = self.query_norm(self.W_query_proj(torch.cat([h_q, vq_dst], dim=1)))

        h_p = self.batch_encode(peer_seqs) if peer_seqs else torch.empty(0, self.dim, device=self.device)

        if self.training and len(aug1) > 0:
            h1 = self.batch_encode(aug1)
            h2 = self.batch_encode(aug2)
            z1 = self.query_norm(self.W_query_proj(torch.cat([h1, vq_dst], dim=1)))
            z2 = self.query_norm(self.W_query_proj(torch.cat([h2, vq_dst], dim=1)))

            p1 = self.self_pred(self.self_proj(z1))
            p2 = self.self_pred(self.self_proj(z2))
            z1_tgt = self.self_proj(z1).detach()
            z2_tgt = self.self_proj(z2).detach()

            def neg_cos(p, z):
                p = F.normalize(p, dim=1)
                z = F.normalize(z, dim=1)
                return 1.0 - (p * z).sum(dim=1)

            loss_self = 0.5 * neg_cos(p1, z2_tgt).mean() + 0.5 * neg_cos(p2, z1_tgt).mean()
            self.aux_loss = self.aux_loss + self.lambda_self * loss_self

        if self.training and h_p.size(0) > 1:
            zp = F.normalize(h_p, dim=1)
            pt = torch.tensor(peer_t, dtype=torch.float, device=self.device)
            py = torch.tensor(peer_y, dtype=torch.long, device=self.device)

            sim_cos = zp @ zp.T
            stage = torch.exp(-self.beta_stage * torch.abs(pt.view(-1, 1) - pt.view(1, -1)))
            sim = (sim_cos * stage) / self.tau_scl

            mask = ~torch.eye(sim.size(0), dtype=torch.bool, device=self.device)

            loss_scl = 0.0
            valid = 0
            for i in range(sim.size(0)):
                pos = (py == py[i]) & mask[i]
                if pos.sum() == 0:
                    continue
                den = torch.logsumexp(sim[i][mask[i]], dim=0)
                mean_pos = sim[i][pos].mean()
                loss_scl = loss_scl + (den - mean_pos)
                valid += 1

            if valid > 0:
                self.aux_loss = self.aux_loss + self.lambda_scl * (loss_scl / valid)

        fused = []
        ptr = 0
        pw = torch.stack(peer_w).to(self.device) if h_p.size(0) > 0 else None

        for i in range(B):
            cnt = peer_counts[i]
            if cnt > 0:
                peers = h_p[ptr:ptr + cnt]
                w = pw[ptr:ptr + cnt].view(-1)

                q_proj = self.W_gat(z_q[i]).unsqueeze(0).expand(cnt, -1)
                k_proj = self.W_gat(peers)

                e = F.leaky_relu(self.a_gat(torch.cat([q_proj, k_proj], dim=1))).squeeze(-1)
                att = F.softmax(e + w, dim=0).unsqueeze(-1)

                fused.append((att * k_proj).sum(dim=0))
                ptr += cnt
            else:
                fused.append(torch.zeros(self.dim, device=self.device))

        fused = torch.stack(fused)
        h_final = F.relu(self.W_prime(z_q) + fused)

        logits = self.predictor(h_final).squeeze(-1)
        return logits


In [ ]:
# Cell SPEED-QCACHE-1 
def _reakt_amp_dtype(self):
    if (str(self.device).startswith("cuda") and torch.is_autocast_enabled()):
        try:
            return torch.get_autocast_dtype("cuda")
        except Exception:
            return torch.get_autocast_gpu_dtype()
    return torch.float32

def question_repr_amp_safe(self, q_ids_global):
    """
    AMP-safe question repr.
    """
    dtype = _reakt_amp_dtype(self)

    qg = torch.as_tensor(q_ids_global, dtype=torch.long, device=self.device).view(-1)
    ql = qg - self.q_offset

    out = torch.zeros(qg.size(0), self.dim, device=self.device, dtype=dtype)

    mask_known = (ql >= 0) & (ql < self.num_questions)
    if mask_known.any():
        ql_valid = ql[mask_known]

        cache = getattr(self, "_qrepr_cache", None)
        if cache is not None:
            out[mask_known] = cache[ql_valid].to(dtype)
        else:
            e_sem = self.sem_proj(self.q_sem[ql_valid])
            e_con = self.con_proj(self.q_con[ql_valid])
            rhs = self.W_s(e_sem) + self.W_c(e_con)
            out[mask_known] = rhs.to(dtype)

    idx_unknown = torch.where(~mask_known)[0]
    if idx_unknown.numel() > 0:
        C = int(self.q_con.shape[1])
        for idx in idx_unknown.tolist():
            q_id = int(qg[idx].item())
            sem_np, cset = self.cgc.get_sem_and_concepts(q_id)
            if sem_np is None:
                continue

            sem_t = torch.from_numpy(np.asarray(sem_np, dtype=np.float32)).to(self.device).view(1, -1)
            con_t = torch.zeros((1, C), device=self.device, dtype=torch.float32)
            for cid in (cset or []):
                ci = int(cid)
                if 0 <= ci < C:
                    con_t[0, ci] = 1.0

            e_sem = self.sem_proj(sem_t)
            e_con = self.con_proj(con_t)
            rhs = self.W_s(e_sem) + self.W_c(e_con)
            out[idx] = rhs.view(-1).to(dtype)

    return out

def batch_encode_amp_safe(self, list_of_seqs):
    """
    AMP-safe batch encoder.
    Benefits strongly from question_repr cache above.
    """
    dtype = _reakt_amp_dtype(self)

    lengths = [len(s[0]) for s in list_of_seqs]
    max_len = max(lengths) if lengths else 0
    if max_len == 0:
        return torch.zeros(len(list_of_seqs), self.dim, device=self.device, dtype=dtype)

    B = len(list_of_seqs)
    x = torch.zeros(B, max_len, self.dim * 2, device=self.device, dtype=dtype)

    for i, (qs, rs) in enumerate(list_of_seqs):
        if len(qs) == 0:
            continue
        vq = self.question_repr(qs).to(dtype)
        eres = self.resp_embed(torch.tensor(rs, dtype=torch.long, device=self.device)).to(dtype)
        x[i, :len(qs), :] = torch.cat([vq, eres], dim=1)

    lengths_cpu = torch.tensor([max(1, l) for l in lengths], dtype=torch.long).cpu()
    packed = nn.utils.rnn.pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=False)
    _, h = self.gru(packed)
    return h[-1]

ReAKT.question_repr = question_repr_amp_safe
ReAKT.batch_encode  = batch_encode_amp_safe



In [ ]:
# Cell SPEED-QCACHE-2

AUX_BALANCE = True
AUX_TARGET_RATIO = 0.5
AUX_SCALE_CLAMP = (0.05, 50.0)

AMP_ENABLED = (device == "cuda")
try:
    _SCALER = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
except Exception:
    _SCALER = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

def _autocast_ctx():
    return torch.autocast(device_type="cuda", enabled=AMP_ENABLED) if AMP_ENABLED else torch.autocast(device_type="cpu", enabled=False)

def _is_index_array(x):
    return isinstance(x, np.ndarray)

def _iter_index_batches(indices: np.ndarray, batch_size: int):
    n = len(indices)
    for st in range(0, n, batch_size):
        yield indices[st:st + batch_size]

@torch.no_grad()
def _build_qrepr_cache(model):
    """
    Build [num_questions, dim] cache in current autocast dtype.
    """
    dtype = torch.get_autocast_dtype("cuda") if (AMP_ENABLED and torch.is_autocast_enabled()) else torch.float32
    qrepr = model.W_s(model.sem_proj(model.q_sem)) + model.W_c(model.con_proj(model.q_con))
    return qrepr.to(dtype)

def evaluate_split(model, eval_data, eval_idx_loader, train_data, split_name="Val", batch_size=256):
    model.eval()
    model.set_memory_mode("val" if split_name.lower().startswith("val") else "test")
    model.reset_state()

    train_order = np.argsort(train_data.node_interact_times)
    train_ptr = 0
    train_tau_sorted = train_data.node_interact_times[train_order]

    preds, labs = [], []
    loss_sum = 0.0
    n_sum = 0
    bce_sum = nn.BCEWithLogitsLoss(reduction="sum")

    with torch.no_grad():
        if _is_index_array(eval_idx_loader):
            idx_sorted = eval_idx_loader.astype(np.int64, copy=False)
            total = math.ceil(len(idx_sorted) / batch_size)
            pbar = tqdm(_iter_index_batches(idx_sorted, batch_size), total=total, desc=f"Eval {split_name}", leave=False)

            for idx_batch in pbar:
                for j in idx_batch:
                    s = int(eval_data.src_node_ids[j])
                    q = int(eval_data.dst_node_ids[j])
                    t = float(eval_data.node_interact_times[j])
                    y = int(eval_data.labels[j])

                    while train_ptr < len(train_order) and float(train_tau_sorted[train_ptr]) < t:
                        k = int(train_order[train_ptr])
                        model.cgc.update(
                            [int(train_data.src_node_ids[k])],
                            [int(train_data.dst_node_ids[k])],
                            [float(train_data.node_interact_times[k])],
                            [int(train_data.labels[k])],
                        )
                        train_ptr += 1

                    with _autocast_ctx():
                        model._qrepr_cache = _build_qrepr_cache(model)
                        logit = model([s], [q], [t], labels=None)
                        model._qrepr_cache = None

                    target = torch.tensor([float(y)], device=logit.device, dtype=logit.dtype)
                    loss_sum += bce_sum(logit.view(-1), target).item()
                    n_sum += 1

                    p = torch.sigmoid(logit).item()
                    preds.append(p)
                    labs.append(y)

                    model.cgc.update([s], [q], [t], [y])
        else:
            for batch_idx in tqdm(eval_idx_loader, desc=f"Eval {split_name}", leave=False):
                idx = batch_idx.numpy()
                idx = idx[np.argsort(eval_data.node_interact_times[idx])]

                for j in idx:
                    s = int(eval_data.src_node_ids[j])
                    q = int(eval_data.dst_node_ids[j])
                    t = float(eval_data.node_interact_times[j])
                    y = int(eval_data.labels[j])

                    while train_ptr < len(train_order) and float(train_tau_sorted[train_ptr]) < t:
                        k = int(train_order[train_ptr])
                        model.cgc.update(
                            [int(train_data.src_node_ids[k])],
                            [int(train_data.dst_node_ids[k])],
                            [float(train_data.node_interact_times[k])],
                            [int(train_data.labels[k])],
                        )
                        train_ptr += 1

                    with _autocast_ctx():
                        model._qrepr_cache = _build_qrepr_cache(model)
                        logit = model([s], [q], [t], labels=None)
                        model._qrepr_cache = None

                    target = torch.tensor([float(y)], device=logit.device, dtype=logit.dtype)
                    loss_sum += bce_sum(logit.view(-1), target).item()
                    n_sum += 1

                    p = torch.sigmoid(logit).item()
                    preds.append(p)
                    labs.append(y)

                    model.cgc.update([s], [q], [t], [y])

    preds = np.asarray(preds, dtype=np.float64)
    labs  = np.asarray(labs, dtype=np.float64)

    auc = roc_auc_score(labs, preds)
    ap  = average_precision_score(labs, preds)
    acc = accuracy_score(labs, (preds > 0.5))
    rmse = float(np.sqrt(np.mean((preds - labs) ** 2)))
    val_loss = float(loss_sum / max(n_sum, 1))

    model.last_eval_loss = val_loss

    print(f"Evaluation Results ({split_name}): AUC={auc:.4f}  AP={ap:.4f}  ACC={acc:.4f}  RMSE={rmse:.4f}  Loss={val_loss:.6f}")
    return auc, ap, acc, rmse


def train_reakt(
    question_sem_features,
    question_concept_multi_hot,
    q_offset,
    question_node_ids_all,
    q_emb,
    q_concept_sets_for_cgc,
    train_data, val_data, test_data,
    train_idx_loader, val_idx_loader,
    epochs=100,
    lr=1e-3,
    dim=128,
    concept_dim=None,
    sem_cb=None,
    concept_cb=None,
    batch_size=256,
    early_stopping=True,
    patience=5,
    min_delta=1e-4,
):
    mem_train = StudentMemoryIndex(train_data, num_users)
    mem_val   = StudentMemoryIndex(val_data,   num_users)
    mem_test  = StudentMemoryIndex(test_data,  num_users)

    cgc = ContextGraphCache(
        question_node_ids=question_node_ids_all,
        question_embeddings=q_emb,
        question_concept_sets=q_concept_sets_for_cgc,
        device=device,
        time_decay=TIME_DECAY,
        window_size=W_WINDOW,
        prune_interval=PRUNE_INTERVAL,
        epsilon_ratio=EPS_RATIO,
        edge_min_w=EDGE_MIN_W,
        max_degree=MAX_DEGREE,
        max_anchors=MAX_ANCHORS,
        build_block=CGC_BUILD_BLOCK,
        concept_dim=concept_dim,
        sem_cb=sem_cb,
        concept_cb=concept_cb,
        candidate_factor=6,
        activate_at_init=False,
    ).to(device)

    model = ReAKT(
        question_sem_features=question_sem_features,
        question_concept_multi_hot=question_concept_multi_hot,
        q_offset=q_offset,
        cgc=cgc,
        mem_train=mem_train,
        mem_val=mem_val,
        mem_test=mem_test,
        device=device,
        dim=dim,
        lambda_scl=LAMBDA_SCL,
        lambda_self=LAMBDA_SELF,
        tau_scl=TAU_SCL,
        beta_stage=BETA_STAGE,
        max_seq_len=MAX_SEQ_LEN,
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    bce = nn.BCEWithLogitsLoss()

    try:
        scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

    best_val_loss = float("inf")
    best_state = None
    bad_epochs = 0

    print(f"Starting ReAKT training (AMP={AMP_ENABLED}, qrepr-cache=ON)")

    for ep in range(epochs):
        model.train()
        model.set_memory_mode("train")
        model.reset_state()

        if _is_index_array(train_idx_loader):
            idx_sorted = train_idx_loader.astype(np.int64, copy=False)
            total = math.ceil(len(idx_sorted) / batch_size)
            pbar = tqdm(_iter_index_batches(idx_sorted, batch_size), total=total, desc=f"Epoch {ep+1}/{epochs}")
            for idx in pbar:
                src = train_data.src_node_ids[idx]
                dst = train_data.dst_node_ids[idx]
                tt  = train_data.node_interact_times[idx]
                yy  = torch.tensor(train_data.labels[idx], dtype=torch.float, device=device)

                opt.zero_grad(set_to_none=True)

                with _autocast_ctx():
                    model._qrepr_cache = _build_qrepr_cache(model)
                    logits = model(src, dst, tt, labels=yy)
                    model._qrepr_cache = None

                    loss_pred = bce(logits, yy)
                    aux_raw = model.aux_loss

                    if AUX_BALANCE and aux_raw.detach().abs().item() > 1e-12:
                        scale = (AUX_TARGET_RATIO * loss_pred.detach()) / (aux_raw.detach() + 1e-12)
                        scale = torch.clamp(scale, AUX_SCALE_CLAMP[0], AUX_SCALE_CLAMP[1])
                        aux = aux_raw * scale
                    else:
                        aux = aux_raw

                    loss = loss_pred + aux

                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()

                pbar.set_postfix({
                    "loss": float(loss.item()),
                    "bce": float(loss_pred.item()),
                    "aux_raw": float(aux_raw.item()),
                    "aux_used": float(aux.item()),
                })
        else:
            pbar = tqdm(train_idx_loader, desc=f"Epoch {ep+1}/{epochs}")
            for batch_idx in pbar:
                idx = batch_idx.numpy()
                idx = idx[np.argsort(train_data.node_interact_times[idx])]

                src = train_data.src_node_ids[idx]
                dst = train_data.dst_node_ids[idx]
                tt  = train_data.node_interact_times[idx]
                yy  = torch.tensor(train_data.labels[idx], dtype=torch.float, device=device)

                opt.zero_grad(set_to_none=True)

                with _autocast_ctx():
                    model._qrepr_cache = _build_qrepr_cache(model)
                    logits = model(src, dst, tt, labels=yy)
                    model._qrepr_cache = None

                    loss_pred = bce(logits, yy)
                    aux_raw = model.aux_loss

                    if AUX_BALANCE and aux_raw.detach().abs().item() > 1e-12:
                        scale = (AUX_TARGET_RATIO * loss_pred.detach()) / (aux_raw.detach() + 1e-12)
                        scale = torch.clamp(scale, AUX_SCALE_CLAMP[0], AUX_SCALE_CLAMP[1])
                        aux = aux_raw * scale
                    else:
                        aux = aux_raw

                    loss = loss_pred + aux

                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()

                pbar.set_postfix({
                    "loss": float(loss.item()),
                    "bce": float(loss_pred.item()),
                    "aux_raw": float(aux_raw.item()),
                    "aux_used": float(aux.item()),
                })

        evaluate_split(model, val_data, val_idx_loader, train_data, split_name="Val", batch_size=batch_size)
        val_loss = float(getattr(model, "last_eval_loss", float("inf")))

        if val_loss < (best_val_loss - float(min_delta)) or best_state is None:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if early_stopping and bad_epochs >= int(patience):
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


In [ ]:
# Cell 11: Sanity checks


def build_untrained_reakt_for_sanity(
    question_sem_features,
    question_concept_multi_hot,
    q_offset,
    question_node_ids_all,
    q_emb,
    q_concept_sets_for_cgc,
    train_data, val_data, test_data,
    num_users,
    device,
    dim=128,
):
    mem_train = StudentMemoryIndex(train_data, num_users)
    mem_val   = StudentMemoryIndex(val_data,   num_users)
    mem_test  = StudentMemoryIndex(test_data,  num_users)

    cgc = ContextGraphCache(
        question_node_ids=question_node_ids_all,
        question_embeddings=q_emb,
        question_concept_sets=q_concept_sets_for_cgc,
        device=device,
        time_decay=TIME_DECAY,
        window_size=W_WINDOW,
        prune_interval=PRUNE_INTERVAL,
        epsilon_ratio=EPS_RATIO,
        edge_min_w=EDGE_MIN_W,
        max_degree=MAX_DEGREE,
        max_anchors=MAX_ANCHORS,
    ).to(device)

    model = ReAKT(
        question_sem_features=question_sem_features,
        question_concept_multi_hot=question_concept_multi_hot,
        q_offset=q_offset,
        cgc=cgc,
        mem_train=mem_train,
        mem_val=mem_val,
        mem_test=mem_test,
        device=device,
        dim=dim,
        lambda_scl=LAMBDA_SCL,
        lambda_self=LAMBDA_SELF,
        tau_scl=TAU_SCL,
        beta_stage=BETA_STAGE,
        max_seq_len=MAX_SEQ_LEN,
    ).to(device)

    return model

def check_betas_in_optimizer(model, opt):
    params = [p for g in opt.param_groups for p in g["params"]]
    print("beta0 in opt:", any(id(p) == id(model.cgc.beta0) for p in params))
    print("beta1 in opt:", any(id(p) == id(model.cgc.beta1) for p in params))

@torch.no_grad()
def check_no_future_anchors(model, data_obj, n=500):
    """
    Validates the mandatory causal constraint: every retrieved anchor must satisfy tau_k < tau_t.
    """
    model.eval()
    model.reset_state()

    idx = np.argsort(data_obj.node_interact_times)[:n]
    for k in idx:
        s = int(data_obj.src_node_ids[k])
        q = int(data_obj.dst_node_ids[k])
        t = float(data_obj.node_interact_times[k])

        anchors = model.cgc.get_batch_anchors([q], [t])[0]
        for a in anchors:
            assert float(a["timestamp"]) < t, (a["timestamp"], t)

        model.cgc.update([s], [q], [t], [int(data_obj.labels[k])])

    print(f"PASS: no anchors with tau_k >= tau_t in first {n} events")

reakt_model = build_untrained_reakt_for_sanity(
    question_sem_features=question_sem_features,
    question_concept_multi_hot=question_concept_multi_hot,
    q_offset=Q_OFFSET,
    question_node_ids_all=question_node_ids_all,
    q_emb=q_emb,
    q_concept_sets_for_cgc=q_concept_sets_for_cgc,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    num_users=num_users,
    device=device,
    dim=128,
)

check_no_future_anchors(reakt_model, train_data, n=500)


@torch.no_grad()
def check_no_future_anchors(model, data_obj, n=500):
    model.eval()
    model.reset_state()

    idx = np.argsort(data_obj.node_interact_times)[:n]
    for k in idx:
        s = int(data_obj.src_node_ids[k])
        q = int(data_obj.dst_node_ids[k])
        t = float(data_obj.node_interact_times[k])

        anchors = model.cgc.get_batch_anchors([q], [t])[0]
        for a in anchors:
            assert float(a["timestamp"]) < t, (a["timestamp"], t)

        model.cgc.update([s], [q], [t], [int(data_obj.labels[k])])

    print(f"PASS: no anchors with tau_k >= tau_t in first {n} events")

@torch.no_grad()
def check_stage1_cached_neighbors(model, data_obj, n=200, max_degree=MAX_DEGREE):
    model.eval()
    model.reset_state()

    idx = np.argsort(data_obj.node_interact_times)[:n]
    for k in idx:
        s = int(data_obj.src_node_ids[k])
        q = int(data_obj.dst_node_ids[k])
        t = float(data_obj.node_interact_times[k])

        (anchors, dbg) = model.cgc.get_batch_anchors([q], [t], return_debug=True)
        examined = dbg[0]["stage1_examined"]

        ql_t = model.cgc.qid2local.get(int(q), None)
        if ql_t is None:
            expected_examined = 0
        else:
            expected_examined = len(model.cgc.adj[int(ql_t)])

        assert examined == expected_examined, (examined, expected_examined)
        if max_degree > 0:
            assert examined <= max_degree, (examined, max_degree)

        model.cgc.update([s], [q], [t], [int(data_obj.labels[k])])

    print(f"PASS: Stage-1 iterates only over cached neighbors (first {n} events)")


check_stage1_cached_neighbors(reakt_model, train_data, n=200, max_degree=MAX_DEGREE)
check_no_future_anchors(reakt_model, train_data, n=500)

opt = torch.optim.Adam(reakt_model.parameters(), lr=1e-3)
params = [p for g in opt.param_groups for p in g["params"]]
print("beta0 in opt:", any(id(p) == id(reakt_model.cgc.beta0) for p in params))
print("beta1 in opt:", any(id(p) == id(reakt_model.cgc.beta1) for p in params))


@torch.no_grad()
def check_active_graph_invariants(model, sample_inactive=2000):
    cgc = model.cgc

    act = np.where(cgc.active)[0]
    for ql in act.tolist():
        assert int(ql) in cgc.buffers and len(cgc.buffers[int(ql)]) > 0

    inact = np.where(~cgc.active)[0]
    if inact.size > sample_inactive:
        inact = np.random.choice(inact, size=sample_inactive, replace=False)
    for ql in inact.tolist():
        assert int(ql) not in cgc.buffers
        assert len(cgc.adj[int(ql)]) == 0

    print("PASS: active/inactive CGC invariants")

@torch.no_grad()
def check_node_removed_when_buffer_expires(model, train_data):
    cgc = model.cgc
    model.reset_state()

    q0 = int(train_data.dst_node_ids[np.argmin(train_data.node_interact_times)])
    q1 = q0
    for x in train_data.dst_node_ids[:1000]:
        if int(x) != q0:
            q1 = int(x); break
    assert q1 != q0

    cgc.update([0], [q0], [0.0], [1])
    ql0 = int(cgc.qid2local[q0])
    assert cgc.active[ql0]

    T = float(cgc.window_size) + 10.0
    for i in range(int(cgc.prune_interval) + 5):
        cgc.update([0], [q1], [T + i], [1])

    assert not bool(cgc.active[ql0])
    assert len(cgc.adj[ql0]) == 0
    assert ql0 not in cgc.buffers
    print("PASS: stale node deactivated + removed from graph")

    
reakt_model = build_untrained_reakt_for_sanity(
    question_sem_features=question_sem_features,
    question_concept_multi_hot=question_concept_multi_hot,
    q_offset=Q_OFFSET,
    question_node_ids_all=question_node_ids_all,
    q_emb=q_emb,
    q_concept_sets_for_cgc=q_concept_sets_for_cgc,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    num_users=num_users,
    device=device,
    dim=128,
)

reakt_model.reset_state()

check_active_graph_invariants(reakt_model)

check_node_removed_when_buffer_expires(reakt_model, train_data)

for k in np.argsort(train_data.node_interact_times)[:2000]:
    reakt_model.cgc.update(
        [int(train_data.src_node_ids[k])],
        [int(train_data.dst_node_ids[k])],
        [float(train_data.node_interact_times[k])],
        [int(train_data.labels[k])]
    )
check_active_graph_invariants(reakt_model)


In [ ]:
# Cell 12: train CaRe-KT and evaluate
##################################################################
#SET01
reakt_model = train_reakt(
    question_sem_features=question_sem_features,
    question_concept_multi_hot=question_concept_multi_hot,
    q_offset=Q_OFFSET,
    question_node_ids_all=question_node_ids_all,
    q_emb=q_emb,
    q_concept_sets_for_cgc=q_concept_sets_for_cgc,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    train_idx_loader=train_idx_loader,
    val_idx_loader=val_idx_loader,
    epochs=100,
    lr=1e-3,
    dim=64,
    concept_dim=num_concepts,
    sem_cb=SEM_CB,
    concept_cb=CONCEPT_CB,
)

##################################################################
#SET02
# reakt_model = train_reakt(
#     question_sem_features=question_sem_features,
#     question_concept_multi_hot=question_concept_multi_hot,
#     q_offset=Q_OFFSET,
#     question_node_ids_all=question_node_ids_all,
#     q_emb=q_emb,
#     q_concept_sets_for_cgc=q_concept_sets_for_cgc,
#     train_data=train_data,
#     val_data=val_data,
#     test_data=test_data,
#     train_idx_loader=train_idx_loader,
#     val_idx_loader=val_idx_loader,
#     epochs=5,
#     lr=8e-4,
#     dim=160,
#     concept_dim=num_concepts,
#     sem_cb=SEM_CB,
#     concept_cb=CONCEPT_CB,
# )



##################################################################
#SET03

# reakt_model = train_reakt(
#     question_sem_features=question_sem_features,
#     question_concept_multi_hot=question_concept_multi_hot,
#     q_offset=Q_OFFSET,
#     question_node_ids_all=question_node_ids_all,
#     q_emb=q_emb,
#     q_concept_sets_for_cgc=q_concept_sets_for_cgc,
#     train_data=train_data,
#     val_data=val_data,
#     test_data=test_data,
#     train_idx_loader=train_idx_loader,
#     val_idx_loader=val_idx_loader,
#     epochs=5,
#     lr=1.5e-3,
#     dim=96,
#     concept_dim=num_concepts,
#     sem_cb=SEM_CB,
#     concept_cb=CONCEPT_CB,
# )



##################################################################

evaluate_split(reakt_model, test_data, test_idx_loader, train_data, split_name="Test")
